In [ ]:
# Diffusion model dependencies (TabDDPM + CoDi)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# CoDi: ChaejeongLee/CoDi (_vendor/CoDi)
%pip install -q ForestDiffusion xgboost category-encoders libzero rtdl imbalanced-learn absl-py tensorboardX

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "goggle" / "src"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "CoDi"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_codi


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
secondary_mushroom = fetch_ucirepo(id=848)

# data (as pandas dataframes)
X = secondary_mushroom.data.features
y = secondary_mushroom.data.targets

# metadata
print(secondary_mushroom.metadata)

# variable information
print(secondary_mushroom.variables)

mushroom_data = pd.concat([X, y], axis=1)

target_col = "class"

CONTINUOUS_COLS = ["cap-diameter", "stem-height", "stem-width"]
CATEGORICAL_COLS = [
    col for col in mushroom_data.columns if col not in CONTINUOUS_COLS
]

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

# Fast dev mode: fewer epochs for generators only (set False for full paper run)
FAST_MODE = True
TabDDPM_EPOCHS = 10 if FAST_MODE else 150
WGAN_EPOCHS = 10 if FAST_MODE else 100
SDV_EPOCHS = 10 if FAST_MODE else 300
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
# All 6 synthetic generators for TSTR evaluation
GENERATORS_TO_EVAL = [
    "CTGAN", "CopulaGAN", "TVAE", "GaussianCopula", "CoDi", "TabDDPM"
]

# Stratified subsample (full dataset has 61069 rows)
_, mushroom_data = train_test_split(
    mushroom_data,
    train_size=N_SAMPLES,
    stratify=mushroom_data[target_col],
    random_state=SEED,
)
mushroom_data = mushroom_data.reset_index(drop=True)

# Handle missing values
numeric_cols = mushroom_data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = [col for col in CATEGORICAL_COLS if col in mushroom_data.columns]

for col in numeric_cols:
    mushroom_data[col] = pd.to_numeric(mushroom_data[col], errors="coerce")
    mushroom_data[col] = mushroom_data[col].fillna(mushroom_data[col].median())

for col in categorical_cols:
    fill = mushroom_data[col].mode().iloc[0] if not mushroom_data[col].mode().empty else "missing"
    mushroom_data[col] = mushroom_data[col].fillna(fill)

# Label-encode categorical columns (ordinal encoding)
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    mushroom_data[col] = le.fit_transform(mushroom_data[col].astype(str))
    label_encoders[col] = le

# Encoded positive class for binary metrics (poisonous = "p")
pos_label = int(label_encoders[target_col].transform(["p"])[0])

print(f"Encoded {len(categorical_cols)} categorical columns.")
print(f"{target_col} mapping: {dict(zip(label_encoders[target_col].classes_, range(len(label_encoders[target_col].classes_))))}")
print(f"pos_label (encoded poisonous class): {pos_label}")

X = mushroom_data.drop(columns=[target_col])
y = mushroom_data[target_col]

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(mushroom_data)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []


In [ ]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

CONTINUOUS_COLS = ["cap-diameter", "stem-height", "stem-width"]
CATEGORICAL_COLS = [
    col for col in mushroom_data.columns if col not in CONTINUOUS_COLS
]

# Re-apply encoding if the load cell was not re-run after edits
if mushroom_data[target_col].dtype == "object":
    numeric_cols = mushroom_data.select_dtypes(include=["int64", "float64"]).columns
    for col in numeric_cols:
        mushroom_data[col] = pd.to_numeric(mushroom_data[col], errors="coerce")
        mushroom_data[col] = mushroom_data[col].fillna(mushroom_data[col].median())
    for col in CATEGORICAL_COLS:
        fill = mushroom_data[col].mode().iloc[0] if not mushroom_data[col].mode().empty else "missing"
        mushroom_data[col] = mushroom_data[col].fillna(fill)
    label_encoders = {}
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        mushroom_data[col] = le.fit_transform(mushroom_data[col].astype(str))
        label_encoders[col] = le
    pos_label = int(label_encoders[target_col].transform(["p"])[0])
elif "label_encoders" not in globals():
    label_encoders = {}
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        le.fit(mushroom_data[col].astype(str))
        label_encoders[col] = le
    try:
        pos_label = int(label_encoders[target_col].transform(["p"])[0])
    except ValueError:
        pos_label = int(mushroom_data[target_col].max())

# ---------------------------------------------------
# GENERATOR TRAINING DATA (no stratified split)
# ---------------------------------------------------

train_real = mushroom_data.copy()
test_real = mushroom_data.copy()

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

if 'TabDDPM' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training TabDDPM...')
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['TabDDPM'] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_metadata,
            )
            scores['TabDDPM'] = quality.get_score()
            print('TabDDPM:', round(scores['TabDDPM'], 4))
        else:
            print('TabDDPM: trained (quality eval skipped)')
    except Exception as e:
        print('TabDDPM Failed:', e)
        traceback.print_exc()
else:
    print('TabDDPM: skipped (not in GENERATORS_TO_EVAL)')


In [ ]:
# CoDi
if 'CoDi' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training CoDi...')
        synthetic_codi = train_codi(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['CoDi'] = synthetic_codi.copy()
        print('CoDi: synthesis complete')
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_codi,
                metadata=train_metadata,
            )
            scores['CoDi'] = quality.get_score()
            print('CoDi:', round(scores['CoDi'], 4))
        else:
            print('CoDi: trained (quality eval skipped)')
    except Exception as e:
        print('CoDi Failed (training/sampling):')
        traceback.print_exc()
    if 'CoDi' in synthetic_datasets and RUN_QUALITY_EVAL:
        pass
else:
    print('CoDi: skipped (not in GENERATORS_TO_EVAL)')


In [ ]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "TVAE": TVAESynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata),
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

if FAST_MODE:
    models = {
        "LogReg": LogisticRegression(max_iter=500, solver="liblinear", random_state=42),
        "SVM-RBF": LinearSVC(max_iter=500, dual="auto", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42, max_depth=12),
        "RandomForest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "GradientBoost": GradientBoostingClassifier(
            n_estimators=30, random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
        "MLP": MLPClassifier(max_iter=200, random_state=42),
    }
else:
    models = {
        "LogReg": LogisticRegression(max_iter=5000, solver="liblinear", random_state=42),
        "SVM-RBF": SVC(kernel="rbf", cache_size=1000, tol=1e-3, random_state=42),
        "KNN": KNeighborsClassifier(n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42),
    }

print(f"Classifier evaluation: {len(models)} models, {len(EVAL_SEEDS)} seeds")
if FAST_MODE:
    print("FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.")


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import pandas as pd


In [ ]:
# TRTR is evaluated in the comparison cell below via evaluate_models().
print(
    "Skipping duplicate TRTR cell. "
    f"Run the comparison cell for TRTR/TSTR ({len(models)} models, {len(EVAL_SEEDS)} seeds)."
)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np


def _safe_stratify(y):
    y = pd.Series(y).reset_index(drop=True)
    if y.nunique() < 2 or y.value_counts().min() < 2:
        return None
    return y


def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None,
):
    if seeds is None:
        seeds = EVAL_SEEDS

    results = []

    for name, model in models.items():
        print(f"  {name}...", flush=True)

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=_safe_stratify(y_train),
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=_safe_stratify(y_test),
            )

            scaler = StandardScaler().fit(X_train)
            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)
            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)
            if hasattr(clf, "n_jobs"):
                clf.set_params(n_jobs=-1)

            clf.fit(X_train_s, y_train)
            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(
                f1_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )
            precision_scores.append(
                precision_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )
            recall_scores.append(
                recall_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )

        results.append({
            "Model": name,
            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),
            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),
            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),
            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),
            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by="Accuracy Mean", ascending=False)


In [ ]:
import pandas as pd

label_col = "class"

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "CoDi",
    "TabDDPM"
]

seeds = EVAL_SEEDS

print("TRTR (Train Real, Test Real)")
print(
    f"Classifiers: {len(models)} | Seeds: {len(seeds)} | "
    f"Generators: {len(model_order)}"
)
print(f"Classifier models: {list(models.keys())}")
print(f"Synthetic generators: {model_order}")

trtr_results = evaluate_models(
    train_df=mushroom_data,
    test_df=mushroom_data,
    label="class",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=mushroom_data,
        label="class",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


In [ ]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")
